[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Connection Pools


## What you will be able to do

Open a pool with either driver, take a connection from it and give it back, and say what the pool
did that you did not have to. Size it, and say where the number comes from, which is the database
server and not your traffic. Watch a pool survive every one of its connections being closed
underneath it, and watch the same pool fail when it is not asked to check. Find a leaked connection
from the exception it causes. And say where a pool belongs in a program, which is once, at startup.


## The idea

### The problem

Opening a PostgreSQL connection is not cheap. There is a TCP handshake, possibly a TLS handshake,
authentication, and then the server forks a process for you. Doing that inside a request handler
adds all of it to every request, and doing it under load adds it hundreds of times a second to a
server that is already busy.

The other half of the problem is the opposite one. Connections are not free to the server either,
and a program that opens one per concurrent request will at some point open more than the server
allows and take the whole thing down. A pool is what sits between those two failures.

### What a pool is

A fixed number of connections, opened once, handed out one at a time and taken back. Your code
borrows one for the length of a query or a transaction and returns it, and the pool does the
opening, the closing, the replacing of broken ones, and the waiting when they are all busy.

### Why it works that way

Because a connection is a server process, the right number of them is a property of the server. The
PostgreSQL project's own guidance sizes a pool from the machine's cores rather than from how many
clients you have, and more connections past that point make throughput worse rather than better: the
work queues up either way, and it queues more cheaply in your pool than in the server's scheduler.

### Where this shows up

Every service, and the two failures above are both common. A pool per request is the first one
written out longhand. Two services with pools of twenty against a server that allows fifty is the
second one, and it is an outage rather than a slowdown.

### What this notebook covers

Both pools: `psycopg_pool.ConnectionPool`, its async twin, and `asyncpg.create_pool`. Sizing, and
where the number comes from. What happens when the server closes every connection the pool holds,
with and without `check`. Then the leak, the timeout it causes, the two tasks that share one
connection, and where in a program a pool is opened.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio
import time

import asyncpg


async def main():
    pool = await asyncpg.create_pool(database="guide", min_size=2, max_size=2)

    async def one_request(number):
        async with pool.acquire() as conn:              # wait for a turn, then hand it back
            return await conn.fetchval("SELECT pg_sleep(0.2), $1::int", number)

    start = time.perf_counter()
    await asyncio.gather(*(one_request(number) for number in range(6)))
    print(f"6 requests through 2 connections: {time.perf_counter() - start:.1f}s")
    print("three turns of two, and not one connection was opened to serve them")

    await pool.close()


asyncio.run(main())
```

```
6 requests through 2 connections: 0.6s
three turns of two, and not one connection was opened to serve them
```

Six requests, two connections, three turns, and the arithmetic comes out exactly. The waiting is
real and it is visible in the timing, and that is the trade a pool makes: a request may wait for a
connection, and in exchange no request ever pays to open one.


## Setup

Thirteen imports, both pools, the server, and two helpers.

- `psycopg_pool` is a separate package from `psycopg`, which is why it is named in the install
- `asyncpg` brings its own pool, and `exceptions` is where its classes live
- `asyncio`, `time` and `json` run and measure the examples, and `logging` quiets the pool's retries
- `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and `PackageNotFoundError`

`terminate` closes every backend wearing one `application_name`, which is how the sections below
stage a server restart without restarting anything. `backends` counts the ones open now, and
`sessions` counts every connection the server has ever accepted, which is how the last section
counts what opening a pool costs.


In [1]:
import asyncio
import getpass
import json
import logging
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
import psycopg_pool
from asyncpg import exceptions

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

logging.getLogger("psycopg.pool").setLevel(logging.CRITICAL)        # its retries are not the lesson


def terminate(application_name):
    """Close every backend wearing one application_name, which is a server restart in miniature."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        return conn.execute("SELECT count(pg_terminate_backend(pid)) FROM pg_stat_activity "
                            "WHERE application_name = %s", (application_name,)).fetchone()[0]


def backends(application_name):
    """How many connections the server currently has under that name."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        return conn.execute("SELECT count(*) FROM pg_stat_activity WHERE application_name = %s",
                            (application_name,)).fetchone()[0]


def sessions():
    """How many connections the server has accepted since it started, counting from PostgreSQL 14."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:                # this one included
        return conn.execute("SELECT sessions FROM pg_stat_database "
                            "WHERE datname = 'guide'").fetchone()[0]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


## Worked examples

### A psycopg pool, opened and used

`open=True` opens it now, `wait` blocks until the first connections are actually there, and
`pool.connection()` is the block you borrow inside:


In [2]:
pool = psycopg_pool.ConnectionPool("dbname=guide application_name=first_pool",
                                   min_size=2, max_size=4, open=True)
pool.wait(timeout=10)                                               # or find out later, mid request

with pool.connection() as conn:
    print("inside the block:", conn.execute("SELECT count(*) FROM events").fetchone())

print("the connection is still open after the block:", not conn.closed)
print("the server has", backends("first_pool"), "connections under this pool's name")


inside the block: (5000,)
the connection is still open after the block: True
the server has 2 connections under this pool's name


Two things differ from a plain `with psycopg.connect(...)` block, and both matter. Leaving the block
commits, as it always did, but it does not close the connection: it hands it back. And the pool
opened `min_size` connections before any of this ran, which is what `wait` was waiting for.

`get_stats` is how you find out what a pool is doing in a running program:


In [3]:
for key, value in sorted(pool.get_stats().items()):
    if not key.endswith("_ms") and not key.endswith("_num"):
        print(f"  {key}: {value}")

with pool.connection():
    print("  while one is checked out, pool_available:", pool.get_stats()["pool_available"])


  pool_available: 2
  pool_max: 4
  pool_min: 2
  pool_size: 2
  requests_waiting: 0
  while one is checked out, pool_available: 1


`requests_waiting` is the number to put on a dashboard. It is zero here, and anything else in
production means the pool is too small for the work or a connection is not coming back.

### Sizing it

The number comes from the database server, not from you:


In [4]:
with psycopg.connect("dbname=guide") as conn:
    allowed = int(conn.execute("SHOW max_connections").fetchone()[0])

cores = os.cpu_count()
print(f"this machine reports {cores} cores, and the server allows {allowed} connections")
print(f"a starting point of (cores * 2) + 1 is {cores * 2 + 1}")
print("that is a starting point and not an answer: measure, then change it")


this machine reports 14 cores, and the server allows 100 connections
a starting point of (cores * 2) + 1 is 29
that is a starting point and not an answer: measure, then change it


The PostgreSQL project's guidance is that throughput stops improving past roughly twice the core
count, and the often-cited HikariCP result is a pool cut from two thousand connections to under a
hundred, with response times falling by a large factor rather than rising. Both say the same thing:
size the pool to the server, then tune.

The second number is the one that causes outages. Every pool in every process shares the server's
one `max_connections`, so two services with pools of twenty against a server that allows fifty work
until both are busy at once, and then neither can connect. Add your pools up before you set them.

### An asyncpg pool

Same idea, different spelling, and one extra thing worth having:


In [5]:
async def register_json(conn):
    """Run on every connection the pool opens, which is what init is for."""
    await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")


apool = await asyncpg.create_pool(database="guide", min_size=2, max_size=4, init=register_json)

async with apool.acquire() as conn:
    payload = await conn.fetchval("SELECT payload FROM events ORDER BY id LIMIT 1")
    print("the codec was registered for us:", type(payload).__name__, payload)

print("shortcut, with no acquire at all:", await apool.fetchval("SELECT count(*) FROM events"))
print("size:", apool.get_size(), "| idle:", apool.get_idle_size(), "| max:", apool.get_max_size())


the codec was registered for us: dict {'n': 1, 'size': 2}
shortcut, with no acquire at all: 5000
size: 2 | idle: 2 | max: 4


`init` answers the problem **asyncpg** left open: a type codec is per connection, and a pool opens
connections whenever it likes. `init` is the hook that runs on each one.

The shortcut is worth knowing: `pool.fetchval`, `pool.fetch`, `pool.execute` all acquire, run and
release in one call. Use them for a single statement, and `acquire` when several statements have to
be on the same connection, which includes every transaction.

### Surviving the server closing every connection

This is the failure a pool exists to absorb. Here it is staged with `pg_terminate_backend`, which is
what a restart does to your connections, without restarting anything:


In [6]:
unchecked = psycopg_pool.ConnectionPool("dbname=guide application_name=unchecked",
                                        min_size=2, max_size=2, open=True)
unchecked.wait(timeout=10)

print("terminated", terminate("unchecked"), "of the pool's connections")
time.sleep(0.3)                                                     # let the server finish

try:
    with unchecked.connection() as conn:
        conn.execute("SELECT 1")
except psycopg.OperationalError as error:
    print("the next request:", type(error).__name__ + ":", str(error).splitlines()[0])


terminated 2 of the pool's connections
the next request: AdminShutdown: terminating connection due to administrator command


The pool handed out a connection it was still holding and the server had already closed. Nothing was
wrong with the pool's bookkeeping; it simply had no way to know.

One argument fixes it:


In [7]:
checked = psycopg_pool.ConnectionPool("dbname=guide application_name=checked",
                                      min_size=2, max_size=2, open=True,
                                      check=psycopg_pool.ConnectionPool.check_connection)
checked.wait(timeout=10)

print("terminated", terminate("checked"), "of the pool's connections")
time.sleep(0.3)

with checked.connection() as conn:
    print("the next request:", conn.execute("SELECT 1").fetchone(), "<- it replaced them")
print("the server has", backends("checked"), "connections under this pool's name again")


terminated 2 of the pool's connections
the next request: (1,) <- it replaced them
the server has 2 connections under this pool's name again


`check` runs before every hand-out, so a dead connection is discovered by the pool rather than by
your request. It costs one round trip per borrow, which is the cheapest insurance in this notebook.

A connection that was closed underneath you can be recognized directly:


In [8]:
single = psycopg_pool.ConnectionPool("dbname=guide application_name=single",
                                     min_size=1, max_size=1, open=True)
single.wait(timeout=10)
borrowed = single.getconn()

terminate("single")
time.sleep(0.3)
try:
    borrowed.execute("SELECT 1")
except psycopg.OperationalError:
    print("closed:", borrowed.closed, "| broken:", borrowed.broken)

single.putconn(borrowed)                                            # the pool discards a broken one
single.close()


closed: True | broken: True


`closed` means this object is finished with. `broken` means it was closed by something other than
you, which is the one worth logging, because it is the server telling you something.

### When to reach for which

| What you want | psycopg | asyncpg |
|---|---|---|
| a pool | `psycopg_pool.ConnectionPool(conninfo, ...)` | `await asyncpg.create_pool(...)` |
| the async one | `psycopg_pool.AsyncConnectionPool` | there is only the async one |
| open it | `open=True`, then `pool.wait()` | the `await` on `create_pool` |
| borrow one | `with pool.connection() as conn` | `async with pool.acquire() as conn` |
| one statement, no borrowing | nothing | `await pool.fetchval(sql)` |
| set every connection up | `configure=` | `init=` |
| notice a connection died | `check=ConnectionPool.check_connection` | it checks on release |
| how big | `min_size`, `max_size` | `min_size`, `max_size` |
| how long to wait for a turn | `timeout=` on the pool | `timeout=` on the pool |
| what it is doing | `pool.get_stats()` | `get_size`, `get_idle_size` |

The default is one pool per process, opened at startup and closed at shutdown, sized from the
database server's cores. Reach for `pool.connection()` or `pool.acquire()` per unit of work, and
never hold one across anything slow that is not the database.

### A service that opens its pool once, finished

The shape a web application has. The lifespan handler is FastAPI's name for it, and the point is
that `create_pool` appears exactly once in the program.


In [9]:
class Service:
    """One pool, opened at startup and closed at shutdown, with handlers that only borrow."""

    def __init__(self, dsn, size):
        self.dsn, self.size, self.pool = dsn, size, None

    async def startup(self):
        self.pool = await asyncpg.create_pool(self.dsn, min_size=1, max_size=self.size,
                                              init=register_json)
        await self.pool.execute("DROP TABLE IF EXISTS intake")
        await self.pool.execute("CREATE TABLE intake (kind text, payload jsonb)")

    async def shutdown(self):
        await self.pool.close()

    async def summary(self):                                        # one statement, no acquire
        rows = await self.pool.fetch("SELECT kind, count(*) AS n FROM events "
                                     "GROUP BY kind ORDER BY kind")
        return [(row["kind"], row["n"]) for row in rows]

    async def record(self, kind, payload):                          # a transaction, so acquire
        async with self.pool.acquire() as conn:
            async with conn.transaction():
                await conn.execute("INSERT INTO intake (kind, payload) VALUES ($1, $2)",
                                   kind, payload)
                return await conn.fetchval("SELECT count(*) FROM intake WHERE kind = $1", kind)


service = Service("postgresql:///guide", size=4)
await service.startup()

print("summary:", await service.summary())
print("recorded, and the new count:", await service.record("click", {"n": 0, "size": 1}))
print("six requests at once:", len(await asyncio.gather(*(service.summary() for _ in range(6)))))

await service.shutdown()


summary: [('click', 1666), ('purchase', 1667), ('view', 1667)]
recorded, and the new count: 1
six requests at once: 6


Two kinds of handler, and the difference is the whole lesson. `summary` runs one statement and uses
the shortcut, so it holds a connection for as short a time as the pool can manage. `record` needs
two statements to be in one transaction, so it borrows explicitly and keeps the borrow for exactly
as long as the transaction lasts.

### Where each part came from

| In the service | What it relies on | The section that showed it |
|---|---|---|
| one `create_pool` | a pool opened once, not per request | A first look |
| `init=register_json` | a codec on every connection the pool opens | An asyncpg pool |
| `await self.pool.fetch(...)` | acquire, run and release in one call | An asyncpg pool |
| `async with self.pool.acquire()` | several statements on one connection | A first look |
| `async with conn.transaction()` | asyncpg commits each statement without it | **asyncpg** |
| `max_size=4` | a number that comes from the server | Sizing it |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/13-connection-pools-solutions.ipynb).

**1.** Open a psycopg pool of two, run one query through it, and close it.


In [10]:
# your code here


**2.** Show how many connections the server has for a pool before and after `wait`.


In [11]:
# your code here


**3.** Run eight slow queries through an asyncpg pool of two and time them.


In [12]:
# your code here


**4.** Take a connection out of a pool of one without returning it, and show what the next borrower
gets.


In [13]:
# your code here


**5.** Terminate a pool's connections and show that a pool with `check` keeps working.


In [14]:
# your code here


**6.** Give an asyncpg pool an `init` that sets `application_name`, and count the backends wearing
it.


In [15]:
# your code here


## Common errors

### psycopg_pool.PoolTimeout: couldn't get a connection after 2.00 sec


In [16]:
leaky = psycopg_pool.ConnectionPool("dbname=guide application_name=leaky",
                                    min_size=1, max_size=1, timeout=2, open=True)
leaky.wait(timeout=10)

held = leaky.getconn()                                              # taken, and never given back
leaky.getconn()


PoolTimeout: couldn't get a connection after 2.00 sec

The pool is not broken and the server is fine. One connection exists, something is holding it, and
everybody else waits until the timeout.

In a real program this is almost never a literal `getconn`. It is a `with pool.connection()` block
that something long-running was put inside, or an exception path that skipped the `putconn`. The
`with` block is the fix, because it returns the connection however the block ends:


In [17]:
leaky.putconn(held)

try:
    with leaky.connection() as conn:
        conn.execute("SELECT 1")
        raise RuntimeError("a request that failed halfway")
except RuntimeError as error:
    print("the handler raised:", error)

with leaky.connection() as conn:
    print("and the connection came back anyway:", conn.execute("SELECT 1").fetchone())
leaky.close()


the handler raised: a request that failed halfway
and the connection came back anyway: (1,)


### psycopg_pool.PoolTimeout: pool initialization incomplete after 2 sec


In [18]:
nowhere = psycopg_pool.ConnectionPool("host=127.0.0.1 port=1 dbname=guide",
                                      min_size=1, open=False)
nowhere.open()
nowhere.wait(timeout=2)


PoolTimeout: pool initialization incomplete after 2 sec

The same class, a different message, and a completely different cause: the pool cannot reach the
server at all. This is what `wait` is for. Without it the pool opens quietly and the failure arrives
inside the first request instead, which is a much worse place to learn that the database is down.

`reconnect_failed` is the callback for the other half of this, a pool that was working and then could
not reconnect, which is where a program raises an alert rather than an exception:


In [19]:
def give_up(pool):
    print(f"  reconnect_failed fired for {pool.name}: page somebody")


watched = psycopg_pool.ConnectionPool("host=127.0.0.1 port=1 dbname=guide", min_size=1,
                                      name="the_watched_pool", reconnect_timeout=2,
                                      reconnect_failed=give_up, open=True)
time.sleep(4)
watched.close()
print("the program kept running, which is the point")


  reconnect_failed fired for the_watched_pool: page somebody
the program kept running, which is the point


### asyncpg.exceptions.InterfaceError: cannot perform operation: another operation is in progress


In [20]:
async with apool.acquire() as conn:
    slow = asyncio.create_task(conn.fetchval("SELECT pg_sleep(0.5), 1"))
    await asyncio.sleep(0.1)                                        # let the first one get going

    try:
        await conn.fetchval("SELECT 2")
    except exceptions.InterfaceError as error:
        print("InterfaceError:", error)

    await slow                                                      # finish before handing it back


InterfaceError: cannot perform operation: another operation is in progress


Two tasks, one acquired connection. **AsyncConnection** showed psycopg quietly serializing this;
asyncpg refuses instead, and the refusal is better, because the psycopg version looks like code that
works and runs at half the speed.

The fix is one `acquire` per task, which is what a pool is for:


In [21]:
async def properly(number):
    async with apool.acquire() as conn:                             # one each
        return await conn.fetchval("SELECT $1::int FROM pg_sleep(0.3)", number)


start = time.perf_counter()
print("two tasks, two connections:", await asyncio.gather(properly(1), properly(2)),
      f"in {time.perf_counter() - start:.1f}s")


two tasks, two connections: [1, 2] in 0.3s


### No error: a pool opened for every request


In [22]:
async def per_request():
    """Correct, and the reason a service is slow."""
    pool = await asyncpg.create_pool(database="guide", min_size=1, max_size=1)
    try:
        return await pool.fetchval("SELECT 1")
    finally:
        await pool.close()


before = sessions()
await asyncio.gather(*(per_request() for _ in range(20)))
a_pool_each = sessions() - before - 1                               # less the connection that asked

before = sessions()
await asyncio.gather(*(apool.fetchval("SELECT 1") for _ in range(20)))
one_pool = sessions() - before - 1

print("20 requests, a pool each:", a_pool_each, "new connections for the server to open")
print("20 requests, one pool:   ", one_pool)
print("both answered 1, and only one of them is a service")


20 requests, a pool each: 20 new connections for the server to open
20 requests, one pool:    2
both answered 1, and only one of them is a service


Nothing failed. Twenty pools were opened, used once and closed, and the server counted every
connection. The second line is the pool growing from its `min_size` of two to its `max_size` of
four, once, and then serving the rest of the twenty with what it already had. That is the number to
look at: twenty requests cost the server two connections, and they would cost it none the next time.

Counting is better than timing here, because over a local Unix socket twenty connections cost a few
milliseconds and the number would look like nothing. Every one of those twenty is a TCP handshake, a
TLS handshake against a real server, an authentication and a forked process, and that is what a
count makes visible where a stopwatch on this machine does not.

This is the most common pooling mistake and it never raises. The shape that prevents it is the one
in the finished example: the pool is created where the program starts, not where the work happens.


In [23]:
for closing in (pool, unchecked, checked):
    closing.close()
for closing in (apool,):
    await closing.close()
print("pools closed")


pools closed


## Recap

- A pool is a fixed number of connections, opened once and handed out. Borrowing is cheap, opening
  is not, and a request may wait for a turn instead.
- `psycopg_pool` is a separate package. `ConnectionPool(open=True)` then `pool.wait()`, and
  `with pool.connection()` commits and hands the connection back without closing it.
- `asyncpg.create_pool` is awaited, `pool.acquire()` borrows, and `pool.fetch` and friends borrow and
  return in one call. `init=` runs on every connection the pool opens.
- Size from the database server's cores, roughly twice the core count as a starting point, and add
  every pool in every process up against `max_connections` before you deploy.
- A pool does not notice that the server closed its connections until it tries. `check=` makes it
  check first, `conn.broken` tells you it was closed by somebody else, and `reconnect_failed` is
  where a program raises an alert.
- `PoolTimeout` means either a leaked connection or a server that was never reachable, and the
  message says which.
- Two tasks on one acquired asyncpg connection raise `InterfaceError`. Acquire one each.
- Opening a pool per request raises nothing and undoes the entire point.


## What is next

**Connecting to a Hosted Server** is the same connections against a server you did not start: a DSN
out of the environment, TLS that actually verifies something, and the connection limit a managed
provider enforces on the role you were given.


---

&#8592; **Previous:** [Prepared Statements](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/12-prepared-statements.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Connecting to a Hosted Server](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/14-connecting-to-a-hosted-server.ipynb) &#8594;
